In [1]:
import sys
import weakref
import gc

## Задание 1

Сформулировать класс, который демонстрирует, как CPython хранит данные экземпляра в словаре и как это связано с атрибутом `__dict__`.

Реализуйте класс TrackedObject, который:
- В конструкторе принимает произвольные именованные аргументы и записывает их в атрибуты экземпляра.
- Переопределяет `__setattr__` и `__delattr__`, чтобы:
  - Логировать каждое изменение в списке history (атрибут экземпляра).
  - Отслеживать реальный размер `__dict__` до и после операции.

In [2]:
class TrackedObject:
  def __init__(self, **kwargs):
    # history сам добавляем через super().__setattr__, чтобы не ловить в логе
    super().__setattr__('history', [])
    for k, v in kwargs.items():
      setattr(self, k, v)

  def __setattr__(self, name, value):
    d = self.__dict__
    size_before = sys.getsizeof(d)
    old = d.get(name, None)
    existed = name in d
    super().__setattr__(name, value)
    size_after = sys.getsizeof(self.__dict__)
    self.history.append({
      "op": "set",
      "name": name,
      "old": old if existed else None,
      "new": value,
      "dict_size_before": size_before,
      "dict_size_after": size_after,
    })

  def __delattr__(self, name):
    d = self.__dict__
    size_before = sys.getsizeof(d)
    old = d[name]
    super().__delattr__(name)
    size_after = sys.getsizeof(self.__dict__)
    self.history.append({
      "op": "del",
      "name": name,
      "old": old,
      "dict_size_before": size_before,
      "dict_size_after": size_after,
    })

obj = TrackedObject(x=1, y=2)
obj.z = 3                # добавление нового атрибута
obj.__str__ = lambda s: "patched"  # перезатирка метода
del obj.x

print(obj.__dict__)
for event in obj.history:
  print(event)

{'history': [{'op': 'set', 'name': 'x', 'old': None, 'new': 1, 'dict_size_before': 296, 'dict_size_after': 296}, {'op': 'set', 'name': 'y', 'old': None, 'new': 2, 'dict_size_before': 296, 'dict_size_after': 296}, {'op': 'set', 'name': 'z', 'old': None, 'new': 3, 'dict_size_before': 296, 'dict_size_after': 296}, {'op': 'set', 'name': '__str__', 'old': None, 'new': <function <lambda> at 0x10d113530>, 'dict_size_before': 296, 'dict_size_after': 296}, {'op': 'del', 'name': 'x', 'old': 1, 'dict_size_before': 296, 'dict_size_after': 296}], 'y': 2, 'z': 3, '__str__': <function <lambda> at 0x10d113530>}
{'op': 'set', 'name': 'x', 'old': None, 'new': 1, 'dict_size_before': 296, 'dict_size_after': 296}
{'op': 'set', 'name': 'y', 'old': None, 'new': 2, 'dict_size_before': 296, 'dict_size_after': 296}
{'op': 'set', 'name': 'z', 'old': None, 'new': 3, 'dict_size_before': 296, 'dict_size_after': 296}
{'op': 'set', 'name': '__str__', 'old': None, 'new': <function <lambda> at 0x10d113530>, 'dict_size_

## Задание 2

Создать нетривиальный ромбовидный и более сложный граф наследования, а затем вручную вывести C3‑линеаризацию и сверить с `__mro__`:

- Постройте иерархию классов не менее чем из 6 классов с несколькими ромбами (несколько общих предков).
- В одной из веток сделайте «конфликт» имён методов (одинаковый метод в двух разных базах).
- Напишите функцию `c3_linearize(cls)`, которая по списку баз реализует алгоритм C3‑линеаризации (без использования внутренностей CPython).
- Для нескольких классов:
  - Выведите результат вашей функции.
  - Выведите `cls.__mro__`.
- Прокомментируйте, почему порядок разрешения методов именно такой, и как C3 гарантирует локальный порядок и отсутствие конфликтов.

In [3]:
def merge(seqs):
  seqs = [list(s) for s in seqs]
  result = []
  while True:
    nonempty = [s for s in seqs if s]
    if not nonempty:
      return result
    candidate = None
    for s in nonempty:
      h = s[0]
      ok = all(h not in t[1:] for t in nonempty)
      if ok:
        candidate = h
        break
    if candidate is None:
      raise TypeError("Cannot merge MRO: inconsistent hierarchy")
    result.append(candidate)
    for s in seqs:
      if s and s[0] == candidate:
        s.pop(0)


def c3_linearize(cls):
  if cls is object:
    return [object]
  bases = list(cls.__bases__)
  seqs = [c3_linearize(b) for b in bases] + [bases]
  return [cls] + merge(seqs)


# Иерархия из 7 классов: два «ромба» через общий корень O.
class O:
  pass


class A(O):
  def f(self):
    return "A"


class B(O):
  def f(self):
    return "B"


class C(A, B):
  pass


class D(A):
  pass


class E(B):
  pass


class F(D, E):
  pass


for cls in (C, F, D):
  manual = tuple(c3_linearize(cls))
  builtin = cls.__mro__
  print(f"{cls.__name__}: manual C3 == __mro__ ? {manual == builtin}")
  print("  c3_linearize:", manual)
  print("  __mro__:", builtin)

# Комментарий:
# C3 строит линеаризацию так, что суперкласс всегда идёт после подклассов (локальный порядок
# в списке баз сохраняется), а общие предки вроде A и B упорядочиваются согласованно во всей
# иерархии — поэтому не возникает противоречий. При «конфликте» имён (f в A и B) вызывается
# та реализация, чей класс раньше в MRO (для F раньше D, чем E, поэтому для F через D идёт A
# перед веткой через E к B).

C: manual C3 == __mro__ ? True
  c3_linearize: (<class '__main__.C'>, <class '__main__.A'>, <class '__main__.B'>, <class '__main__.O'>, <class 'object'>)
  __mro__: (<class '__main__.C'>, <class '__main__.A'>, <class '__main__.B'>, <class '__main__.O'>, <class 'object'>)
F: manual C3 == __mro__ ? True
  c3_linearize: (<class '__main__.F'>, <class '__main__.D'>, <class '__main__.A'>, <class '__main__.E'>, <class '__main__.B'>, <class '__main__.O'>, <class 'object'>)
  __mro__: (<class '__main__.F'>, <class '__main__.D'>, <class '__main__.A'>, <class '__main__.E'>, <class '__main__.B'>, <class '__main__.O'>, <class 'object'>)
D: manual C3 == __mro__ ? True
  c3_linearize: (<class '__main__.D'>, <class '__main__.A'>, <class '__main__.O'>, <class 'object'>)
  __mro__: (<class '__main__.D'>, <class '__main__.A'>, <class '__main__.O'>, <class 'object'>)


## Задание 3

Исследовать, как именно работает name mangling в CPython для «закрытых» атрибутов и как это отражается в `__dict__` и `dir()`.

- Реализуйте класс SecureBase c атрибутами:
  - `__secret_value`
  - `_semi_private`
  - `public`

- Унаследуйте от него класс `SecureChild`, где:
  - Переопределите `__secret_value` и `_semi_private`.
  - Добавьте метод, который возвращает содержимое `self.__dict__`.

- Напишите код, который:
  - Показывает результат `dir()` и `__dict__` для экземпляров обоих классов.
  - Демонстрирует, под какими реальными именами хранятся «закрытые» атрибуты.
  - Пытается получить доступ к «закрытому» атрибуту через сгенерированное имя (`_ИмяКласса__secret_value`).


In [4]:
class SecureBase:
    def __init__(self):
        self.__secret_value = "base_secret"
        self._semi_private = "base_semi"
        self.public = "base_public"


class SecureChild(SecureBase):
    def __init__(self):
        super().__init__()
        self.__secret_value = "child_secret"
        self._semi_private = "child_semi"

    def dump_dict(self):
        return dict(self.__dict__)


base = SecureBase()
child = SecureChild()

print("=== SecureBase instance ===")
print("dir():", [a for a in dir(base) if "secret" in a or "semi" in a or a == "public"])
print("__dict__:", base.__dict__)

print("\n=== SecureChild instance ===")
print("dir():", [a for a in dir(child) if "secret" in a or "semi" in a or a == "public"])
print("__dict__:", child.__dict__)
print("dump_dict():", child.dump_dict())

print("\nAccess mangled from outside:")
print("base._SecureBase__secret_value  =", base._SecureBase__secret_value)
print("child._SecureBase__secret_value =", child._SecureBase__secret_value)
print("child._SecureChild__secret_value=", child._SecureChild__secret_value)
print("child._semi_private             =", child._semi_private)

=== SecureBase instance ===
dir(): ['_SecureBase__secret_value', '_semi_private', 'public']
__dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'base_semi', 'public': 'base_public'}

=== SecureChild instance ===
dir(): ['_SecureBase__secret_value', '_SecureChild__secret_value', '_semi_private', 'public']
__dict__: {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'child_semi', 'public': 'base_public', '_SecureChild__secret_value': 'child_secret'}
dump_dict(): {'_SecureBase__secret_value': 'base_secret', '_semi_private': 'child_semi', 'public': 'base_public', '_SecureChild__secret_value': 'child_secret'}

Access mangled from outside:
base._SecureBase__secret_value  = base_secret
child._SecureBase__secret_value = base_secret
child._SecureChild__secret_value= child_secret
child._semi_private             = child_semi


## Задание 4

Показать влияние `__slots__` на структуру объекта, наличие `__dict__` и возможность динамического добавления атрибутов, а также `weakref`.

- Опишите три класса:
  - NoSlots: без `__slots__`.
  - WithSlots: с `__slots__ = ("x", "y")`.
  - WithSlotsWeak: с `__slots__ = ("x", "__weakref__")`.
- Для каждого класса:
  - Создайте серию экземпляров, замерьте:
    - Наличие `__dict__` и `__weakref__` (через `hasattr` и `dir`).
    - Возможность динамически добавить новый атрибут `z`.
  - Используя модуль sys, оцените примерный размер одного экземпляра (через getsizeof плюс, при наличии, размер `__dict__`).
- Покажите, для каких классов возможно создавать слабые ссылки (`weakref.ref`).

In [5]:
class NoSlots:
    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlots:
    __slots__ = ("x", "y")

    def __init__(self, x, y):
        self.x = x
        self.y = y


class WithSlotsWeak:
    __slots__ = ("x", "__weakref__")

    def __init__(self, x):
        self.x = x


def describe_instance(obj):
    d = {
        "type": type(obj).__name__,
        "has_dict": hasattr(obj, "__dict__"),
        "has_weakref": hasattr(obj, "__weakref__"),
        "size_obj": sys.getsizeof(obj),
    }
    if hasattr(obj, "__dict__"):
        d["size_dict"] = sys.getsizeof(obj.__dict__)
        d["dict_keys"] = list(obj.__dict__.keys())
    else:
        d["size_dict"] = None
        d["dict_keys"] = None
    return d


n = NoSlots(1, 2)
w = WithSlots(1, 2)
ww = WithSlotsWeak(1)

for obj in (n, w, ww):
    print(describe_instance(obj))

print("\n--- Dynamic attribute addition ---")
n.z = 99
print(f"NoSlots: n.z = {n.z}")

for obj, name in ((w, "WithSlots"), (ww, "WithSlotsWeak")):
    try:
        obj.z = 99
        print(f"{name}: obj.z = {obj.z}")
    except AttributeError as e:
        print(f"{name}: cannot add z -> {e}")

print("\n--- weakref support ---")
for obj, name in ((n, "NoSlots"), (ww, "WithSlotsWeak")):
    wr = weakref.ref(obj)
    print(f"{name}: weakref.ref -> {wr()}")

try:
    wr_w = weakref.ref(w)
except TypeError as e:
    print(f"WithSlots: weakref.ref -> {e}")

{'type': 'NoSlots', 'has_dict': True, 'has_weakref': True, 'size_obj': 48, 'size_dict': 296, 'dict_keys': ['x', 'y']}
{'type': 'WithSlots', 'has_dict': False, 'has_weakref': False, 'size_obj': 48, 'size_dict': None, 'dict_keys': None}
{'type': 'WithSlotsWeak', 'has_dict': False, 'has_weakref': True, 'size_obj': 56, 'size_dict': None, 'dict_keys': None}

--- Dynamic attribute addition ---
NoSlots: n.z = 99
WithSlots: cannot add z -> 'WithSlots' object has no attribute 'z' and no __dict__ for setting new attributes
WithSlotsWeak: cannot add z -> 'WithSlotsWeak' object has no attribute 'z' and no __dict__ for setting new attributes

--- weakref support ---
NoSlots: weakref.ref -> <__main__.NoSlots object at 0x10d128ec0>
WithSlotsWeak: weakref.ref -> <__main__.WithSlotsWeak object at 0x10d16c850>
WithSlots: weakref.ref -> cannot create weak reference to 'WithSlots' object


## Задание 5

Исследовать, как слабые ссылки учитываются в подсчёте ссылок и как ведут себя при циклических структурах.

- Определите класс Node, который:
  - Может ссылаться на «родителя» через обычную сильную ссылку.
  - Может ссылаться на «родителя» через weakref.ref.
- Постройте:
  - Циклический граф с сильными ссылками и измерьте:
  - Счётчики ссылок через sys.getrefcount для ключевых объектов.
  - Поведение GC до и после удаления внешних ссылок (модуль gc).
- Аналогичную структуру, но часть ссылок сделайте слабыми.

- Покажите:
  - Что происходит с результатом вызова слабой ссылки после удаления объекта.
  - Как GC обрабатывает циклы со слабыми и без слабых ссылок.

In [6]:
class Node:
    def __init__(self, name, parent=None, weak_parent=False):
        self.name = name
        if weak_parent and parent is not None:
            self._parent = weakref.ref(parent)
        else:
            self._parent = parent
        self._weak = weak_parent

    def get_parent(self):
        if self._weak and self._parent is not None:
            return self._parent()
        return self._parent

    def __repr__(self):
        return f"Node({self.name!r})"


def refcount(obj):
    return sys.getrefcount(obj) - 1  # -1 for the temp ref in getrefcount itself


# --- Цикл со сильными ссылками ---
a = Node("a")
b = Node("b", parent=a)
a._parent = b  # create strong cycle: a -> b -> a

print("Strong cycle refcounts:", refcount(a), refcount(b))

del a, b
unreachable = gc.collect()
print(f"GC collected {unreachable} unreachable objects (strong cycle)\n")


# --- Цикл с weakref ---
c = Node("c")
d = Node("d", parent=c, weak_parent=True)  # d -> c via weakref
c._parent = d  # c -> d strong, but d -> c weak => no real cycle for GC

print("Weak cycle refcounts:", refcount(c), refcount(d))
print("d.get_parent():", d.get_parent())

wr_c = weakref.ref(c)
print("wr_c before:", wr_c())

del c
gc.collect()

print("wr_c after deletion:", wr_c())
print("d.get_parent() after c deleted:", d.get_parent())

Strong cycle refcounts: 2 2
GC collected 1787 unreachable objects (strong cycle)

Weak cycle refcounts: 1 2
d.get_parent(): Node('c')
wr_c before: Node('c')
wr_c after deletion: None
d.get_parent() after c deleted: None
